# Validação e Análise da Lotomania

Este notebook valida o CSV de jogos, calcula probabilidades para cada número e gera 15 jogos com 50 números usando heurísticas de frequência e probabilidade.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_FILE = Path('data/Lotomania.csv')

In [2]:
# Carregar e validar dados
print('Arquivo existe:', DATA_FILE.exists())
df = pd.read_csv(DATA_FILE, sep=';', encoding='utf-8-sig', engine='python', parse_dates=['Data Sorteio'], dayfirst=True, keep_default_na=False)
print('Linhas totais:', len(df))
print('Colunas:', list(df.columns)[:10], '...')
print('Última linha:')
print(df.iloc[-1])

# Validações rápidas
assert 'Concurso' in df.columns, 'Coluna Concurso ausente'
assert 'Data Sorteio' in df.columns, 'Coluna Data Sorteio ausente'
assert df['Data Sorteio'].notna().all(), 'Data Sorteio contém valores ausentes'
assert df.filter(regex='^Bola').notna().all().all(), 'Algumas bolas estão ausentes'
print('Validação básica concluída com sucesso.')

Arquivo existe: True
Linhas totais: 2945
Colunas: ['Concurso', 'Data Sorteio', 'Bola1', 'Bola2', 'Bola3', 'Bola4', 'Bola5', 'Bola6', 'Bola7', 'Bola8'] ...
Última linha:
Concurso                                   2945
Data Sorteio                2026-06-29 00:00:00
Bola1                                         0
Bola2                                         4
Bola3                                         5
Bola4                                         7
Bola5                                        10
Bola6                                        14
Bola7                                        18
Bola8                                        22
Bola9                                        24
Bola10                                       40
Bola11                                       46
Bola12                                       47
Bola13                                       57
Bola14                                       59
Bola15                                       64
Bola16         

In [4]:
# Calcular frequência por número
DRAW_COLUMNS = [col for col in df.columns if col.startswith('Bola')]
assert len(DRAW_COLUMNS) == 20, 'Esperado 20 colunas de bolas'

numbers = [f'{n:02d}' for n in range(100)]
all_draws = df[DRAW_COLUMNS].astype(str).apply(lambda col: col.str.zfill(2))
counts = all_draws.values.flatten()
counts_series = pd.Series(counts).value_counts().reindex(numbers, fill_value=0).astype(int)
probabilities = counts_series / counts_series.sum()
summary = pd.DataFrame({'number': numbers, 'count': counts_series.values, 'prob': probabilities.values})
summary = summary.sort_values(['prob', 'count'], ascending=False).reset_index(drop=True)
print(summary.head(20).to_string(index=False))

# Estatísticas gerais
print('Aparições totais:', counts_series.sum())
print('Número mínimo de aparições:', counts_series.min())
print('Número máximo de aparições:', counts_series.max())

number  count     prob
    43    645 0.010951
    47    643 0.010917
    54    634 0.010764
    49    629 0.010679
    24    626 0.010628
    41    625 0.010611
    11    622 0.010560
    48    622 0.010560
    92    622 0.010560
    38    619 0.010509
    31    618 0.010492
    26    617 0.010475
    76    617 0.010475
    44    613 0.010407
    82    613 0.010407
    04    611 0.010374
    42    611 0.010374
    93    611 0.010374
    13    608 0.010323
    64    608 0.010323
Aparições totais: 58900
Número mínimo de aparições: 528
Número máximo de aparições: 645


In [6]:
# Gerar 15 jogos de 50 números
np.random.seed(42)
weights = probabilities.values

def generate_game(seed_offset: int) -> list[str]:
    rng = np.random.default_rng(42 + seed_offset)
    chosen = set()
    candidates = np.array(numbers)
    candidate_weights = weights.copy()
    while len(chosen) < 50:
        candidate_weights = candidate_weights / candidate_weights.sum()
        pick = rng.choice(candidates, p=candidate_weights)
        chosen.add(pick)
        mask = np.isin(candidates, list(chosen), invert=True)
        candidates = candidates[mask]
        candidate_weights = candidate_weights[mask]
    return sorted(chosen, key=int)

games = [generate_game(i) for i in range(15)]
for idx, game in enumerate(games, 1):
    print(f"Jogo {idx:02d}: {', '.join(game)}")

Jogo 01: 03, 05, 09, 10, 11, 12, 14, 17, 18, 21, 23, 25, 27, 28, 31, 33, 34, 35, 40, 41, 42, 43, 44, 45, 47, 53, 59, 61, 62, 63, 64, 66, 68, 70, 72, 73, 74, 77, 78, 79, 81, 82, 83, 84, 86, 88, 92, 95, 96, 97
Jogo 02: 00, 01, 04, 05, 07, 09, 10, 11, 12, 13, 14, 15, 23, 26, 27, 28, 29, 30, 31, 32, 35, 36, 41, 42, 43, 44, 45, 47, 49, 52, 54, 55, 57, 58, 60, 62, 64, 65, 66, 73, 75, 82, 84, 85, 89, 92, 93, 94, 95, 99
Jogo 03: 01, 07, 08, 12, 13, 16, 17, 18, 19, 20, 21, 22, 26, 27, 28, 29, 30, 31, 35, 36, 41, 42, 46, 50, 52, 53, 54, 61, 62, 65, 66, 68, 70, 74, 75, 76, 77, 81, 83, 85, 86, 87, 88, 90, 92, 94, 95, 96, 97, 98
Jogo 04: 00, 02, 03, 04, 05, 10, 11, 13, 14, 15, 19, 21, 22, 31, 32, 33, 35, 37, 40, 44, 46, 47, 48, 51, 56, 57, 58, 61, 64, 65, 66, 67, 68, 69, 70, 71, 73, 76, 77, 78, 79, 81, 82, 84, 85, 87, 95, 96, 97, 99
Jogo 05: 00, 03, 04, 06, 07, 08, 09, 12, 15, 18, 19, 21, 24, 25, 26, 27, 32, 37, 40, 41, 43, 44, 45, 46, 47, 48, 49, 50, 52, 53, 58, 61, 62, 63, 64, 67, 69, 70, 73, 74,

In [7]:
# Resumo dos melhores números por probabilidade
print('Top 50 números por probabilidade:')
print(summary.head(50).to_string(index=False))

from collections import Counter
counter = Counter(number for game in games for number in game)
freq = pd.DataFrame(sorted(counter.items(), key=lambda item: int(item[0])), columns=['number', 'selected_count'])
freq['prob'] = freq['number'].map(summary.set_index('number')['prob'])
print('\nFrequência de seleção nos 15 jogos (cada número pode aparecer entre 0 e 15 vezes):')
print(freq[freq['selected_count'] > 0].sort_values(['selected_count', 'prob'], ascending=[False, False]).head(50).to_string(index=False))

Top 50 números por probabilidade:
number  count     prob
    43    645 0.010951
    47    643 0.010917
    54    634 0.010764
    49    629 0.010679
    24    626 0.010628
    41    625 0.010611
    11    622 0.010560
    48    622 0.010560
    92    622 0.010560
    38    619 0.010509
    31    618 0.010492
    26    617 0.010475
    76    617 0.010475
    44    613 0.010407
    82    613 0.010407
    04    611 0.010374
    42    611 0.010374
    93    611 0.010374
    13    608 0.010323
    64    608 0.010323
    06    607 0.010306
    61    607 0.010306
    67    607 0.010306
    85    607 0.010306
    53    605 0.010272
    17    601 0.010204
    29    601 0.010204
    99    601 0.010204
    87    598 0.010153
    91    598 0.010153
    19    597 0.010136
    45    597 0.010136
    51    597 0.010136
    68    597 0.010136
    83    597 0.010136
    95    597 0.010136
    74    596 0.010119
    90    596 0.010119
    05    595 0.010102
    73    595 0.010102
    78    593 0.010068


In [8]:
from data_loader import LotomaniaDataLoader
from preprocessing import load_cached_draws, preprocess_draws
from feature_engineering import build_feature_matrix
from graph_engine import build_graph_metrics
from markov_engine import MarkovEngine
from hmm_engine import HMMEngine
from ml_engine import MachineLearningEngine
from ensemble_engine import EnsembleEngine
from evaluation_engine import evaluate_last_draw

loader = LotomaniaDataLoader(DATA_FILE)
raw_draws = loader.load_data()
clean_draws = preprocess_draws(raw_draws)
cached = load_cached_draws(clean_draws)
if cached is not None:
    clean_draws = cached

# Build full pipeline metrics
print('Draws loaded:', len(clean_draws))
graph_metrics = build_graph_metrics(clean_draws)
markov_report = MarkovEngine(clean_draws).build_all_orders()
hmm_report = HMMEngine(clean_draws).train_and_score()
features = build_feature_matrix(
    clean_draws,
    graph_metrics=graph_metrics,
    markov_report=markov_report,
    hmm_report=hmm_report,
    include_last_draw=True,
)

ml_next = MachineLearningEngine(clean_draws, features)
ml_next_report = ml_next.predict_next_draw()
ensemble_next = EnsembleEngine(features, ml_next_report, ml_next.model_weights_).build_stack(fit_targets=False)

print('Ensemble predictions computed')
print('Top 20 numbers by prob_final:')
print(ensemble_next[['number', 'prob_final']].head(20).to_string(index=False))

# Generate 15 games of 50 numbers using ensemble probabilities
import numpy as np
numbers = ensemble_next['number'].astype(str).tolist()
weights = ensemble_next['prob_final'].astype(float).to_numpy()
weights = np.clip(weights, 0, None)
if weights.sum() <= 0:
    weights = np.ones_like(weights)

np.random.seed(42)
games = []
for g in range(15):
    rng = np.random.default_rng(42 + g)
    pick = rng.choice(numbers, size=50, replace=False, p=weights / weights.sum())
    games.append(sorted(pick, key=lambda x: int(x)))
    print(f'Jogo {g+1:02d}: {", ".join(sorted(pick, key=lambda x: int(x)))}')


ModuleNotFoundError: No module named 'dotenv'